# Ficheiro 1 - Atividade de Internamento Hospitalar

# Exploração dos dados (ler CSV e explorar)

In [ ]:

import pandas as pd

df = pd.read_csv("../data/raw/atividade-de-internamento-hospitalar_fich1.csv",
                  sep=";", encoding="utf-8-sig")

# Renomear colunas para snake_case sem acentos/espaços
df.columns = ["periodo", "regiao","instituicao","localizacao","especialidade","doentes_saidos","dias_internamento"]


print("Informações do DataFrame")
print("")

print("Dimensão do DataFrame (linhas, colunas):")
print(df.shape)          # (linhas, colunas)
print("")


print("Primeiras 5 linhas: ")
display(df.head())         # primeiras 5 linhas
print("")

print("Estatísticas básicas: ")
display(df.describe().round(2))     # estatísticas básicas das colunas numéricas
print("")

print("Valores em falta por coluna:")
print(df.isna().sum())   # quantos valores em falta por coluna
print("")

print("Verificar se há linhas duplicadas: ")
print(df.duplicated().sum())
print("")


print("Informações: ")
display(df.info())  # colunas, tipos e não nulos
print("")

print("Regiões:")
print(df["regiao"].unique())            # Ver que regiões são abordadas
print("")

print("Insituições:")
for inst in sorted(df["instituicao"].unique()):    # Ver que instituições são abordadas
    print(inst)


# Limpeza e Transformação dos dados
## Preparação do ficheiro para análise:

In [ ]:
# Guardar o nº inicial de linhas, para comparar no fim:
ini = len(df)   

# Eliminar linhas onde ambos "doentes_saidos" e "dias_internamento" são nulos:
df = df[~(df["doentes_saidos"].isnull() & df["dias_internamento"].isnull())]

print("Eliminadas linhas onde ambos 'doentes_saidos' e 'dias_internamento' são nulos.")
print(f"Número linhas inicial: {ini}")
print("Depois:", len(df), "linhas")
print(f"Linhas eliminadas: {ini - len(df)}")
print("")

#-------------------------------------------------------------------------------------------------

# Separar coordenadas da coluna "localização_geográfica" em latitude e longitude
df[["latitude", "longitude"]] = df["localizacao"].str.split(",", expand=True)

# Converter para número
df["latitude"] = df["latitude"].astype(float)
df["longitude"] = df["longitude"].astype(float)

# Eliminar coluna original após separação
df = df.drop(columns=["localizacao"])

print("Latitude e longitude extraídas da coluna 'localizacao':")
print(df[["latitude", "longitude"]].head())
print("")

#-------------------------------------------------------------------------------------------------

# Uniformizar nomes regiões e instituições:

df["regiao"] = df["regiao"].replace(
    "Região de Saúde Norte",
    "Região de Saúde do Norte")


df["instituicao"] = df["instituicao"].replace(
    "EPE",
    "E.P.E.")


#-------------------------------------------------------------------------------------------------

# Calcular demora média apenas onde doentes_saidos > 0
df["demora_media"] = (df["dias_internamento"] / df["doentes_saidos"]).where(df["doentes_saidos"] > 0)

print("Demora média calculada (dias_internamento / doentes_saidos):")
print(df[["doentes_saidos", "dias_internamento", "demora_media"]].head())
print("")

#-------------------------------------------------------------------------------------------------

# Exportar ficheiro tratado
df.to_csv("../data/processed/atividade_de_internamento_tratado.csv", index=False)
print("Ficheiro exportado com sucesso!")
